# 06 Baseline Model

Fast, explainable Logistic Regression baseline yielding calibrated class probabilities.

### 1. Baseline Logistic Regression Training & Evaluation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("../data/processed/employee_attrition_processed.csv")
# Engineer features
df['IncomePerYearAtCompany'] = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1.0)
df['PromotionLagRatio'] = df['YearsSinceLastPromotion'] / (df['YearsInCurrentRole'] + 1.0)
df['TotalSatisfactionScore'] = df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction'] + df['WorkLifeBalance']
df['ExperienceRatio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1.0)

X = df.drop(columns=['EmployeeID', 'Attrition'])
y = (df['Attrition'] == 'Yes').astype(int)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

baseline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

baseline.fit(X_train, y_train)
probs = baseline.predict_proba(X_test)[:, 1]
preds = baseline.predict(X_test)

print("--- Baseline Logistic Regression ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, probs):.4f}")
print(classification_report(y_test, preds))